# 05 — PP-StructureV2 Table Extraction Evaluation

Evaluates the accuracy of PP-StructureV2 table detection on the processed dataset.


In [ ]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/numera-ml")

from pathlib import Path
import json
import numpy as np
from collections import Counter

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/numera-ml/data")
PROCESSED_DIR = DATA_DIR / "processed"


In [ ]:
# Load table detection results
table_results_dir = PROCESSED_DIR / "table_extractions"
results = []
for f in sorted(table_results_dir.glob("*.json")):
    with open(f) as fh:
        results.append(json.load(fh))

print(f"Loaded {len(results)} table extraction results")


In [ ]:
# Compute stats: tables per document, avg rows/cols, confidence distribution
tables_per_doc = [len(r.get("tables", [])) for r in results]
all_confidences = []
all_rows = []
all_cols = []

for r in results:
    for t in r.get("tables", []):
        all_confidences.append(t.get("confidence", 0))
        all_rows.append(t.get("rows", 0))
        all_cols.append(t.get("cols", 0))

print(f"Total tables detected: {sum(tables_per_doc)}")
print(f"Avg tables/doc: {np.mean(tables_per_doc):.1f}")
print(f"Avg confidence: {np.mean(all_confidences):.3f}")
print(f"Avg rows: {np.mean(all_rows):.1f}, Avg cols: {np.mean(all_cols):.1f}")


In [ ]:
# Filter out false-positive tables (too small: less than 2x2)
valid_tables = [t for r in results for t in r.get("tables", [])
                if t.get("rows", 0) >= 2 and t.get("cols", 0) >= 2]
print(f"Valid tables: {len(valid_tables)}")
print(f"Filtered out: {sum(tables_per_doc) - len(valid_tables)} false positives")
